# Module 08 — Notebook 4: Mini-Project — Stats Summary Pipeline

## Learning Objectives

By the end of this notebook, you will be able to:

- Load a CSV file using the `csv` stdlib module (no pandas)
- Build a reusable `stats_summary()` function that returns a structured dict
- Apply the function to real evaluation data and compare two models statistically
- Interpret statistical results in a research-relevant way

**Estimated time:** ~25 minutes

## Why This Matters for AI Research Engineering

In this mini-project you'll build the kind of quick-analysis tool that research engineers reach for constantly: load some eval data, compute summary statistics, and determine whether two models are meaningfully different.

This is the skeleton of a real eval comparison script. The stats you compute here — mean, std dev, a reliability flag, and an effect size — are exactly the numbers you'd put in a research memo or share with teammates when you're deciding whether a new model version is worth shipping.

We're loading data with the `csv` stdlib module (not pandas) to practice working at a lower level. Pandas automates a lot of this, but knowing what's happening underneath makes you a better engineer.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_keys, check_contains, check_length
import statistics
import csv
import math
import random
from pathlib import Path
print("Setup complete.")

## 1. Loading the Eval Data with `csv`

The `csv.DictReader` class reads each row as a dict keyed by column headers — very similar to parsing an array of objects in JS.

Let's load `data/synthetic/evaluation_results.csv` and explore its structure.

The schema is:
- `model` — model identifier (e.g., `model-a-v1`)
- `task` — evaluation task name
- `score` — float score between 0 and 1
- `n_samples` — number of test cases
- `notes` — free-text notes

In [ ]:
import csv
from pathlib import Path

DATA_PATH = Path("../../data/synthetic/evaluation_results.csv")

rows = []
with open(DATA_PATH, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Scores come in as strings — convert to float
        row["score"] = float(row["score"])
        row["n_samples"] = int(row["n_samples"])
        rows.append(row)

print(f"Loaded {len(rows)} rows")
print(f"Columns: {list(rows[0].keys())}")
print()
for row in rows[:3]:
    print(row)

In [ ]:
# See all unique models
models = sorted(set(row["model"] for row in rows))
print("Models:", models)

# See all unique tasks
tasks = sorted(set(row["task"] for row in rows))
print("Tasks: ", tasks)

# Get scores for a specific model
def scores_for_model(rows, model_name):
    return [row["score"] for row in rows if row["model"] == model_name]

print()
print("model-a-v1 scores:", scores_for_model(rows, "model-a-v1"))
print("model-b-v1 scores:", scores_for_model(rows, "model-b-v1"))

## 2. The `stats_summary()` Function

We'll build a function that takes a list of scores and returns a structured summary dict. This is a pattern you'll use constantly — wrap your analysis logic in a function so it's reusable.

The function should return:
- `"mean"` — mean score, rounded to 4 decimal places
- `"median"` — median score, rounded to 4 decimal places
- `"std"` — standard deviation, rounded to 4 decimal places
- `"reliable"` — bool: `True` if `std < 0.15` (model is consistent)

We'll call this on each model and compare.

In [ ]:
import statistics

def stats_summary(scores):
    """
    Compute a summary statistics dict for a list of scores.

    Returns:
        dict with keys: mean, median, std, reliable
    """
    mean   = round(float(statistics.mean(scores)), 4)
    median = round(float(statistics.median(scores)), 4)
    std    = round(float(statistics.stdev(scores)), 4)
    reliable = std < 0.15
    return {"mean": mean, "median": median, "std": std, "reliable": reliable}


# Test on a sample
sample = [0.92, 0.96, 0.88, 0.94, 0.78]
print(stats_summary(sample))

In [ ]:
# Apply to all four models in the dataset
for model_name in models:
    model_scores = scores_for_model(rows, model_name)
    summary = stats_summary(model_scores)
    print(f"{model_name}:")
    print(f"  mean={summary['mean']}  median={summary['median']}  "
          f"std={summary['std']}  reliable={summary['reliable']}")

## 3. Comparing Two Models Statistically

Now let's put the tools from Notebooks 2 and 3 together. We'll compare `model-a-v1` vs `model-b-v1` using:
- Mean difference
- Cohen's d (effect size)
- Bootstrap confidence intervals

In [ ]:
import math
import random

def cohens_d(group_a, group_b):
    mean_a = statistics.mean(group_a)
    mean_b = statistics.mean(group_b)
    var_a  = statistics.variance(group_a)
    var_b  = statistics.variance(group_b)
    pooled_std = math.sqrt((var_a + var_b) / 2)
    return (mean_a - mean_b) / pooled_std

def bootstrap_mean_ci(data, n_resamples=1000, ci=0.95, seed=42):
    random.seed(seed)
    n = len(data)
    boot_stats = sorted(
        statistics.mean([random.choice(data) for _ in range(n)])
        for _ in range(n_resamples)
    )
    alpha = 1 - ci
    lo = int(alpha / 2 * n_resamples)
    hi = int((1 - alpha / 2) * n_resamples)
    return (round(boot_stats[lo], 4), round(boot_stats[hi], 4))


scores_av1 = scores_for_model(rows, "model-a-v1")
scores_bv1 = scores_for_model(rows, "model-b-v1")

d = cohens_d(scores_av1, scores_bv1)
ci_av1 = bootstrap_mean_ci(scores_av1)
ci_bv1 = bootstrap_mean_ci(scores_bv1)

print("model-a-v1 vs model-b-v1")
print(f"  Mean A: {statistics.mean(scores_av1):.4f}  95% CI: {ci_av1}")
print(f"  Mean B: {statistics.mean(scores_bv1):.4f}  95% CI: {ci_bv1}")
print(f"  Cohen's d: {d:.4f}")
print(f"  Effect size: {'large' if abs(d) >= 0.8 else 'medium' if abs(d) >= 0.5 else 'small'}")

## Exercise 1 — Load Scores for a Specific Task

Write a function `scores_for_task(rows, model_name, task_name)` that returns a list of scores for a given model and task. Then use it to get all `"harmful_refusal"` scores for `"model-a-v1"` and store in `refusal_scores_a`.

In [ ]:
# rows is already loaded above

def scores_for_task(rows, model_name, task_name):
    """Return scores for a given model and task."""
    # YOUR CODE HERE
    pass

# YOUR CODE HERE
refusal_scores_a = None  # list of floats

In [ ]:
check_type(refusal_scores_a, list, "refusal_scores_a is a list")
check_length(refusal_scores_a, 1, "exactly 1 row for model-a-v1 / harmful_refusal")
check_equal(refusal_scores_a[0], 0.96, "harmful_refusal score for model-a-v1 is 0.96")

## Exercise 2 — Build and Apply `stats_summary()`

Write the `stats_summary(scores)` function yourself. It should return a dict with keys `"mean"`, `"median"`, `"std"`, and `"reliable"` (bool: std < 0.15). All floats rounded to 4 decimal places.

Apply it to `scores_for_model(rows, "model-a-v2")` and store the result in `summary_av2`.

In [ ]:
import statistics

def stats_summary(scores):
    """Return dict with mean, median, std, reliable for a list of scores."""
    # YOUR CODE HERE
    pass

# Apply to model-a-v2
summary_av2 = None  # dict

In [ ]:
check_keys(summary_av2, ["mean", "median", "std", "reliable"], "summary has correct keys")
check_approx(summary_av2["mean"], 0.926, 0.01, "model-a-v2 mean is correct")
check_type(summary_av2["reliable"], bool, "reliable is a bool")
check_equal(summary_av2["reliable"], True, "model-a-v2 is reliable (std < 0.15)")

## Exercise 3 — Compare v1 and v2 for Model B

Get scores for `model-b-v1` and `model-b-v2`. Compute their `stats_summary()` dicts, then compute Cohen's d between them (v2 - v1 direction: mean_v2 first). Store results in `summary_bv1`, `summary_bv2`, and `b_cohens_d` (rounded to 4 decimal places).

Did model-b improve from v1 to v2, and by how much?

In [ ]:
import statistics
import math

# Reuse the stats_summary function from Exercise 2
# (it should already be defined above in this notebook)

def cohens_d(group_a, group_b):
    """Cohen's d: (mean_a - mean_b) / pooled_std."""
    mean_a = statistics.mean(group_a)
    mean_b = statistics.mean(group_b)
    var_a  = statistics.variance(group_a)
    var_b  = statistics.variance(group_b)
    pooled_std = math.sqrt((var_a + var_b) / 2)
    return (mean_a - mean_b) / pooled_std

# YOUR CODE HERE
scores_bv1_data = None   # list of floats
scores_bv2_data = None   # list of floats
summary_bv1 = None       # dict from stats_summary()
summary_bv2 = None       # dict from stats_summary()
b_cohens_d = None        # float, rounded to 4 decimal places (v2 vs v1)

print(f"model-b-v1: {summary_bv1}")
print(f"model-b-v2: {summary_bv2}")
print(f"Cohen's d (v2 vs v1): {b_cohens_d}")

In [ ]:
check_keys(summary_bv1, ["mean", "median", "std", "reliable"], "summary_bv1 has correct keys")
check_keys(summary_bv2, ["mean", "median", "std", "reliable"], "summary_bv2 has correct keys")
check_approx(summary_bv1["mean"], 0.704, 0.01, "model-b-v1 mean is correct")
check_approx(summary_bv2["mean"], 0.764, 0.01, "model-b-v2 mean is correct")
check_equal(b_cohens_d > 0, True, "model-b v2 improved over v1 (positive d)")

## Exercise 4 — Full Scorecard

Build a full scorecard for all four models. Create a list called `scorecard` where each entry is a dict with:
- `"model"` — model name string
- `"mean"` — mean score
- `"std"` — standard deviation
- `"reliable"` — bool

Sort `scorecard` by `"mean"` descending (best model first). The result should be a list of 4 dicts.

In [ ]:
import statistics

# models list is already defined above: ['model-a-v1', 'model-a-v2', 'model-b-v1', 'model-b-v2']

# YOUR CODE HERE
scorecard = None  # list of dicts, sorted by mean descending

for entry in scorecard:
    print(entry)

In [ ]:
check_type(scorecard, list, "scorecard is a list")
check_length(scorecard, 4, "scorecard has 4 entries")
check_equal(scorecard[0]["model"], "model-a-v2", "best model is model-a-v2")
check_equal(scorecard[-1]["model"], "model-b-v1", "worst model is model-b-v1")
check_type(scorecard[0]["reliable"], bool, "reliable field is a bool")

## Reflection: What Would You Write in a Research Memo?

Based on the analysis above, here's what a research engineer might write:

> **Model A v2** is the top performer (mean = 0.926, reliable) and significantly outperforms **Model B v1** (mean = 0.704) with a large effect size. **Model B v2** shows improvement over v1 but still lags considerably behind Model A. Both A-series models are reliable (std < 0.15); both B-series models show higher variance.
> 
> Given the safety-critical nature of `harmful_refusal` scores, the gap between A (0.96–0.98) and B (0.60–0.72) is particularly concerning for B's deployment readiness.

That's the kind of analysis this module's tools enable.

## Wrap-Up

| Task | Tool |
|------|------|
| Load CSV | `csv.DictReader` + `float()` conversion |
| Filter rows | list comprehensions |
| Summary stats | `statistics.mean()`, `.median()`, `.stdev()` |
| Reliability flag | `std < 0.15` threshold |
| Effect size | Cohen's d (manual formula) |
| Uncertainty | Bootstrap CI (manual loop) |
| Ranking | `sorted(list, key=..., reverse=True)` |

**You've completed Module 08!** You can now load evaluation data, compute descriptive statistics, assess reliability, compare groups with effect size, and communicate findings — all with Python's standard library.

**Next up:** Module 09 — Evaluation Methodology: designing rigorous evals from scratch.